# 🧠 Deep Dive: Building a Transformer from Scratch in PyTorch

This notebook abandons high-level libraries like HuggingFace `transformers`. We are going **Deep Down into AIML**.

We will build the mathematical core of an LLM: **Multi-Head Self-Attention** and the **Transformer Block** using raw PyTorch tensors. If an interviewer asks you how an LLM actually works under the hood, this is the exact math.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

print(f"PyTorch Version: {torch.__version__}")
# We set a manual seed so our random math matrices are exactly the same every time we run this.
torch.manual_seed(42)

## 1. The Core Innovation: Scaled Dot-Product Attention
Before Transformers (2017), AI read text one word at a time. The **Attention Mechanism** allows the AI to look at *every* word in a sentence simultaneously and figure out which words are related to each other.

The equation is: $Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$

- **Q (Query):** What am I looking for?
- **K (Key):** What do I contain?
- **V (Value):** If you match with my Key, here is the information I give you.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(SelfAttention, self).__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads  # Split the embeddings across multiple heads

        assert (self.head_dim * heads == embed_size), "Embed size needs to be divisible by heads"

        # These are the trainable Neural Network weights that learn how to create Queries, Keys, and Values
        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
        
        # The final output layer
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)

    def forward(self, values, keys, query, mask):
        # Get number of training examples
        N = query.shape[0]
        
        # Length of the sequence (e.g., number of words in the sentence)
        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        # Split the embedding into self.heads pieces
        values = values.reshape(N, value_len, self.heads, self.head_dim)
        keys = keys.reshape(N, key_len, self.heads, self.head_dim)
        queries = query.reshape(N, query_len, self.heads, self.head_dim)

        # Pass through the Linear layers to get the actual Q, K, V
        values = self.values(values)
        keys = self.keys(keys)
        queries = self.queries(queries)

        # -----------------------------------------------------
        # CORE MATH: Q * K-Transpose
        # torch.einsum is a highly optimized way to do Matrix Multiplication across multiple dimensions
        # -----------------------------------------------------
        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])

        # Apply the mask (Crucial for text generation so the AI cant look into the future words!)
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        # -----------------------------------------------------
        # CORE MATH: Softmax & Multiply by Value
        # We divide by the square root of head_dim to keep gradients stable (Scaled Dot-Product)
        # -----------------------------------------------------
        attention = torch.softmax(energy / (self.embed_size ** (1 / 2)), dim=3)
        
        # Multiply attention scores by the Values
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
            N, query_len, self.heads * self.head_dim
        )

        # Push through the final neural network layer
        out = self.fc_out(out)
        return out

## 2. The Transformer Block
Self-Attention is just one piece of the puzzle. A full "Transformer Block" consists of:
1. **Self-Attention**
2. **Layer Normalization:** Keeps the mathematical numbers from exploding.
3. **Feed Forward Network:** A standard dense neural network that processes the relationships found by Attention.
4. **Residual Connections (Skip Connections):** Adds the original input back to the output to prevent the vanishing gradient problem.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_size, heads, dropout, forward_expansion):
        super(TransformerBlock, self).__init__()
        
        # 1. The Attention mechanism we built above
        self.attention = SelfAttention(embed_size, heads)
        
        # 2. Layer Normalization (Applied twice)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        # 3. The Feed Forward Neural Network
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion * embed_size, embed_size),
        )

        # 4. Dropout (Prevents overfitting by randomly turning off 10% of neurons during training)
        self.dropout = nn.Dropout(dropout)

    def forward(self, value, key, query, mask):
        # Step 1: Self-Attention
        attention = self.attention(value, key, query, mask)
        
        # Step 2: Skip Connection & LayerNorm (Add & Norm)
        # We add the original query to the attention output!
        x = self.dropout(self.norm1(attention + query))
        
        # Step 3: Feed Forward
        forward = self.feed_forward(x)
        
        # Step 4: Skip Connection & LayerNorm (Add & Norm)
        out = self.dropout(self.norm2(forward + x))
        
        return out

## 3. Testing our AIML Architecture
Let's create a fake tensor of data (representing a batch of 2 sentences, 8 words each) and pass it through our Transformer Block to see the raw math in action!

In [ ]:
# Hyperparameters
embed_size = 256        # Size of the word embeddings
heads = 8               # Number of attention heads
dropout = 0.1           # 10% dropout rate
forward_expansion = 4   # Multiply embed_size by 4 in the feed-forward network

# Instantiate our custom AI architecture
block = TransformerBlock(embed_size, heads, dropout, forward_expansion)

# Create a fake tensor simulating a batch of 2 sentences, each with 8 words (tokens)
# Shape: (Batch_Size, Sequence_Length, Embedding_Size)
fake_data = torch.randn((2, 8, embed_size))

# Pass it through the block! 
# In self-attention, the Value, Key, and Query are all the exact same data!
out = block(fake_data, fake_data, fake_data, mask=None)

print("Success! Our Transformer Block processed the data.")
print(f"Input shape: {fake_data.shape}")
print(f"Output shape: {out.shape}")